In [ ]:
# pip install youtube-transcript-api

# step 1a- Indexing (Document Ingestion)

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

video_id = "Gfr50f6ZBvo"

try:
    api = YouTubeTranscriptApi()
    transcript_obj = api.fetch(video_id=video_id, languages=["en"])

    # Use .text attribute instead of ["text"]
    transcript = " ".join(snippet.text for snippet in transcript_obj)

    print(transcript)  # full concatenated text

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

An unexpected error occurred: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=Gfr50f6ZBvo! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you ha

In [5]:
transcript_obj

NameError: name 'transcript_obj' is not defined

In [ ]:
type(transcript_obj)

# Step b -Indexing(Text Splitting)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
    )
chunks=splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
chunks[0]

# step 1c and 1d - Indexing(Embedding Generation and Storing in Vector Store)

In [ ]:
# embedding
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("Hugging_face_api_token")
hf_embeddings_api = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",  # lightweight
    # model="Qwen/Qwen3-Embedding-8B",
    # task="feature-extraction",# heavyweight but uses a lot of quota token
    huggingfacehub_api_token=api_key,
)


In [ ]:
from langchain_community.vectorstores import Chroma

# Create persistent Chroma vector store (SAVED TO DISK)
vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=hf_embeddings_api,
    collection_names=f"yt_{video_id}",
    persist_directory="./Kristal_chroma_db" # save to this folder in disk and not in RAM
)
# Explicitly persist to disk (ensures data is saved)
vector_store.persist()

print(f"✅ Chroma vector store created with {vector_store._collection.count()} vectors")
print(f"💾 Permanently saved to: {persist_directory}")
print(f"📁 Collection name: yt_{video_id}")